# Groundwater Gaussian Process Regression

## Purpose
Interpolate the MacDonald et al. (2012) categorical groundwater productivity map
into a continuous, uncertainty-quantified surface using Gaussian Process (GP)
regression, then overlay CPIS locations to identify which systems correlate with
groundwater availability.

## Narrative context
This notebook is **Step 4** in the water source attribution arc:

1. *CPIS expansion* — `Code/1_analyze_data`
2. *NDWI activity* — `5_ndwi_analysis.ipynb`
3. *Dam accessibility* — `6_dem_flow_analysis.ipynb`
4. **Groundwater correlation ← This notebook**
   If CPIS are not near serviceable dams, they are likely drawing on groundwater.
   The MacDonald et al. (2012) map is categorical and spatially coarse. GP regression
   produces a smooth surface that respects spatial autocorrelation and quantifies
   prediction uncertainty. High-uncertainty regions identify where additional sampling
   would most reduce model uncertainty.
5. *Anomalies* — `8_anomaly_detection.ipynb`

## Inputs
| Dataset | Config key | Notes |
|---------|-----------|-------|
| Groundwater productivity (processed) | `Groundwater_Prod_gpkg_path` | In repo |
| CPIS polygons (arid SSA) | `SSA_Combined_CPIS_All_shp_path` | In repo |
| Arid SSA boundaries | `SSA_All_by_Country_shp_path` | For visualization |

## Outputs
| File | Config key | Description |
|------|-----------|-------------|
| `GP_Groundwater_Prediction.tif` | `GP_Groundwater_Prediction_tif_path` | Continuous GP mean prediction |
| `GP_Groundwater_Uncertainty.tif` | `GP_Groundwater_Uncertainty_tif_path` | GP uncertainty (1σ) |
| `CPIS_GP_Groundwater.csv` | `CPIS_GP_Groundwater_csv_path` | GP prediction at each CPIS centroid |

## Computational note
GP fitting scales as O(n³) with training points. This notebook subsamples to
`MAX_TRAIN_POINTS` (default 2000) for efficiency. Increase this value with more
available memory, or decrease it for faster iteration during development.

In [ ]:
# --- Import Required Libraries and Utilities ---
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds as transform_from_bounds
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Code.utils.utility import load_config, resolve_path

config = load_config()

## Load and Explore Groundwater Productivity Data

MacDonald et al. (2012) groundwater productivity classes:
- 1 = Low productivity (< 0.1 L/s)
- 2 = Moderate (0.1–1.0 L/s)
- 3 = High (1–10 L/s)
- 4 = Very high (> 10 L/s)
- 0 = No data

These categories are treated as a continuous ordinal variable for GP regression.

In [ ]:
# --- Load groundwater productivity data ---
gw = gpd.read_file(resolve_path(config['Groundwater_Prod_gpkg_path']))
print(f"Groundwater points loaded: {len(gw):,}")
print(f"CRS: {gw.crs}")
print(f"Columns: {list(gw.columns)}")

# Identify the productivity column
prod_col_candidates = [c for c in gw.columns
                       if c.lower() in ('gwprod', 'value', 'prod', 'productivity',
                                        'gw_prod', 'class')]
if not prod_col_candidates:
    # Fall back to first numeric non-geometry column
    prod_col = [c for c in gw.columns
                if gw[c].dtype in (np.float64, np.int64, np.float32) and c != 'geometry'][0]
else:
    prod_col = prod_col_candidates[0]

print(f"\nUsing productivity column: '{prod_col}'")

gw_valid = gw[gw[prod_col] > 0].copy()
print(f"Valid (non-zero) points: {len(gw_valid):,}")
print(f"\nProductivity class distribution:")
print(gw_valid[prod_col].value_counts().sort_index())

In [ ]:
# --- Exploratory visualization ---
arid_ssa = gpd.read_file(resolve_path(config['SSA_All_by_Country_shp_path']))
arid_ssa['geometry'] = arid_ssa['geometry'].simplify(0.01)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
plt.subplots_adjust(wspace=0.1, left=0.05, right=0.95, top=0.88, bottom=0.05)

# Class distribution
gw_valid[prod_col].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_xlabel('Productivity Class', fontsize=14)
axes[0].set_ylabel('Count', fontsize=14)
axes[0].set_title('Groundwater Productivity Class Distribution', fontsize=16)
axes[0].set_xticklabels(
    [f'Class {int(v)}' for v in sorted(gw_valid[prod_col].unique())], rotation=0
)

# Spatial map
arid_ssa.boundary.plot(ax=axes[1], color='black', linewidth=0.8, linestyle='--')
gw_valid.plot(
    column=prod_col, ax=axes[1], cmap='YlOrRd', markersize=1,
    legend=True, legend_kwds={'label': 'Productivity Class', 'orientation': 'vertical'}
)
axes[1].set_title('MacDonald et al. (2012) Groundwater Productivity', fontsize=16)
axes[1].set_axis_off()

fig.suptitle('Groundwater Productivity Data — MacDonald et al. 2012',
             fontsize=20, fontweight='bold', y=0.97)
plt.show()

## Prepare GP Regression Features

We use geographic coordinates (longitude, latitude) as GP input features.
Standardizing coordinates improves numerical stability and avoids sensitivity
to the initial length-scale guess in the RBF kernel.

In [ ]:
# --- Extract features and subsample for GP training ---
MAX_TRAIN_POINTS = 2000  # Increase if memory allows; GP scales as O(n^3)

X_all = np.column_stack([gw_valid.geometry.x, gw_valid.geometry.y])
y_all = gw_valid[prod_col].values.astype(float)

print(f"Total valid points: {len(X_all):,}")

if len(X_all) > MAX_TRAIN_POINTS:
    rng = np.random.default_rng(seed=42)
    idx_train = rng.choice(len(X_all), size=MAX_TRAIN_POINTS, replace=False)
    X_train = X_all[idx_train]
    y_train = y_all[idx_train]
    print(f"Subsampled to {MAX_TRAIN_POINTS:,} training points")
else:
    X_train = X_all
    y_train = y_all
    print(f"Using all {len(X_train):,} points for training")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

print(f"Training X shape: {X_train_sc.shape}")
print(f"y range: [{y_train.min()}, {y_train.max()}]")

## Fit Gaussian Process Regression Model

Kernel: `ConstantKernel × RBF + WhiteKernel`
- `ConstantKernel`: scales the overall signal variance
- `RBF`: captures spatial autocorrelation (length scale = range of correlation)
- `WhiteKernel`: models observation noise / nugget effect

The optimizer uses multiple restarts to avoid local optima in the log-marginal likelihood.

In [ ]:
# --- Fit GP regression ---
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=0.5, noise_level_bounds=(1e-4, 1.0))
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=3,
    random_state=42
)

print("Fitting GP regression model (this may take a minute)...")
gpr.fit(X_train_sc, y_train)

print(f"Optimized kernel: {gpr.kernel_}")
print(f"Log-marginal likelihood: {gpr.log_marginal_likelihood_value_:.2f}")
print(f"R\u00b2 on training set: {gpr.score(X_train_sc, y_train):.3f}")

## Predict on a Regular Grid

We predict GP mean and standard deviation on a regular grid over arid SSA.
Standard deviation (uncertainty) is higher where training data is sparse.

In [ ]:
# --- Predict on a regular grid ---
GRID_RES = 0.25  # degrees; reduce for higher resolution (requires more memory)

bounds = gw_valid.total_bounds  # [xmin, ymin, xmax, ymax]
lon_grid = np.arange(bounds[0], bounds[2], GRID_RES)
lat_grid = np.arange(bounds[1], bounds[3], GRID_RES)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

X_grid = np.column_stack([lon_mesh.ravel(), lat_mesh.ravel()])
X_grid_sc = scaler.transform(X_grid)

print(f"Predicting on {len(X_grid):,} grid points "
      f"({lon_mesh.shape[0]} rows x {lon_mesh.shape[1]} cols)...")
y_pred, y_std = gpr.predict(X_grid_sc, return_std=True)

y_pred_grid = y_pred.reshape(lon_mesh.shape)
y_std_grid = y_std.reshape(lon_mesh.shape)

print(f"Predicted range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
print(f"Uncertainty range: [{y_std.min():.3f}, {y_std.max():.3f}]")

# Save GeoTIFFs
grid_transform = transform_from_bounds(
    lon_grid[0], lat_grid[0], lon_grid[-1], lat_grid[-1],
    lon_mesh.shape[1], lon_mesh.shape[0]
)

for arr, key, name in [
    (y_pred_grid, 'GP_Groundwater_Prediction_tif_path', 'GP mean'),
    (y_std_grid,  'GP_Groundwater_Uncertainty_tif_path', 'GP uncertainty')
]:
    out_path = resolve_path(config[key])
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with rasterio.open(
        out_path, 'w', driver='GTiff', dtype='float32',
        count=1, crs='EPSG:4326',
        transform=grid_transform,
        width=lon_mesh.shape[1], height=lon_mesh.shape[0]
    ) as dst:
        dst.write(np.flipud(arr).astype('float32')[np.newaxis, ...])
    print(f"Saved {name}: {out_path}")

## Visualize GP Prediction vs Categorical Map

Three panels:
- **Left:** GP-predicted groundwater productivity (continuous surface)
- **Center:** GP uncertainty — high values = data-sparse, less reliable predictions
- **Right:** Original MacDonald et al. (2012) categorical map for comparison

The GP reveals gradients hidden in the discrete categories and shows where the
productivity surface is poorly constrained.

In [ ]:
# --- Three-panel comparison ---
extent = [bounds[0], bounds[2], bounds[1], bounds[3]]

fig, axes = plt.subplots(1, 3, figsize=(36, 12))
plt.subplots_adjust(wspace=0.1, left=0.04, right=0.96, top=0.88, bottom=0.05)

# Panel 1: GP mean
im1 = axes[0].imshow(
    np.flipud(y_pred_grid), origin='upper', extent=extent,
    cmap='YlOrRd', vmin=1, vmax=4, aspect='auto'
)
arid_ssa.boundary.plot(ax=axes[0], color='black', linewidth=0.8, alpha=0.7)
plt.colorbar(im1, ax=axes[0], label='Predicted productivity class', fraction=0.04, pad=0.02)
axes[0].set_title('GP Regression: Predicted Groundwater\nProductivity (continuous)', fontsize=18)
axes[0].set_axis_off()

# Panel 2: GP uncertainty
im2 = axes[1].imshow(
    np.flipud(y_std_grid), origin='upper', extent=extent,
    cmap='Blues', vmin=0, aspect='auto'
)
arid_ssa.boundary.plot(ax=axes[1], color='black', linewidth=0.8, alpha=0.7)
plt.colorbar(im2, ax=axes[1], label='Uncertainty (1\u03c3)', fraction=0.04, pad=0.02)
axes[1].set_title('GP Regression: Prediction Uncertainty\n(high = data-sparse regions)', fontsize=18)
axes[1].set_axis_off()

# Panel 3: Categorical map
cmap_cat = {1: '#fee08b', 2: '#fc8d59', 3: '#d73027', 4: '#7f0000'}
for cat_val, color in cmap_cat.items():
    subset = gw_valid[gw_valid[prod_col] == cat_val]
    if len(subset) > 0:
        label = {1: 'Low (1)', 2: 'Moderate (2)', 3: 'High (3)', 4: 'Very High (4)'}[cat_val]
        subset.plot(ax=axes[2], color=color, markersize=1, label=label, alpha=0.7)
arid_ssa.boundary.plot(ax=axes[2], color='black', linewidth=0.8, alpha=0.7)
axes[2].set_title('MacDonald et al. (2012)\nCategorical Map', fontsize=18)
axes[2].set_axis_off()
axes[2].legend(fontsize=12, loc='lower left', markerscale=5)

fig.suptitle('Groundwater Productivity: GP Regression vs. Categorical Map',
             fontsize=24, fontweight='bold', y=0.97)
plt.show()

## Overlay CPIS on GP Prediction Surface

Predict GP groundwater productivity at each CPIS centroid. The scatter-colored
map shows which systems sit in high- vs low-productivity groundwater zones.

In [ ]:
# --- Predict at CPIS centroids and save ---
cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))
cpis['lon'] = cpis.geometry.centroid.x
cpis['lat'] = cpis.geometry.centroid.y

cpis_coords = cpis[['lon', 'lat']].values
cpis_coords_sc = scaler.transform(cpis_coords)
cpis_pred, cpis_std = gpr.predict(cpis_coords_sc, return_std=True)

cpis['gp_gw_productivity'] = cpis_pred
cpis['gp_gw_uncertainty'] = cpis_std

# Save
gp_csv = resolve_path(config['CPIS_GP_Groundwater_csv_path'])
os.makedirs(os.path.dirname(gp_csv), exist_ok=True)
cpis[['lon', 'lat', 'gp_gw_productivity', 'gp_gw_uncertainty']].to_csv(gp_csv)
print(f"CPIS GP values saved: {gp_csv}")

print(f"\nGP productivity at CPIS locations:")
print(f"  Mean:   {cpis_pred.mean():.2f}")
print(f"  Median: {np.median(cpis_pred):.2f}")
print(f"  Range:  [{cpis_pred.min():.2f}, {cpis_pred.max():.2f}]")
print(f"  CPIS in high-productivity areas (predicted class > 2.5): "
      f"{(cpis_pred > 2.5).sum():,} ({100*(cpis_pred > 2.5).mean():.1f}%)")

# Map CPIS on GP surface
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(
    np.flipud(y_pred_grid), origin='upper', extent=extent,
    cmap='YlOrRd', vmin=1, vmax=4, alpha=0.7, aspect='auto'
)
arid_ssa.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.7)
sc = ax.scatter(
    cpis['lon'], cpis['lat'],
    c=cpis['gp_gw_productivity'], cmap='coolwarm_r',
    s=4, vmin=1, vmax=4, alpha=0.8
)
plt.colorbar(sc, ax=ax, label='GP-predicted groundwater productivity class', fraction=0.04)
ax.set_title('CPIS Locations on GP-Predicted Groundwater Productivity',
             fontsize=18, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()